In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from HelperFunctions import read_lesion_monster_file
from pathlib import Path
%load_ext rpy2.ipython

Read in data, set plotting parameters

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")

plt.rcParams.update({
    # Font + text sizes
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  # fallbacks

    # Line widths
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,

    # --- Keep text editable in exports ---
    "svg.fonttype": "none",   # SVG: keep text as <text> elements, not paths
    "pdf.fonttype": 42,       # PDF: embed fonts as TrueType, editable
    "ps.fonttype": 42,        # EPS: same as above
    "text.usetex": False,     # don’t force LaTeX (avoids path-conversion)
})

In [ ]:
root = Path(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Lesion\Lesion')

fullpaths = []

for top in root.iterdir():
    if not top.is_dir():
        continue
    base = top.name
    for leaf in ("Control_1","Control_2","Control_3", "Monster_1", "Monster_2", "Monster_3"):
        sub = top / leaf
        if not sub.is_dir():
            continue
        prefix = f"{base}_{leaf}".lower()

        # use sub.rglob('*') if files can be nested
        for f in sub.iterdir():
            if not f.is_file():
                continue
            name_l = f.name.lower()
            if name_l.startswith(prefix):
                # ensure Control_1 doesn't match Control_10
                n = len(prefix)
                if len(name_l) == n or name_l[n:n+1] in {'_', '-', '.'}:
                    fullpaths.append(str(f))
fullpaths = [p for p in fullpaths
             if not p.lower().endswith(('.csv', '.pickle', '.mp4', '.h5'))]

vsl = ("Gauguin", "Warhol", "Renoir", "Fragonard", "Turner", "Pollock", "Klimt", "Rembrandt", "Nessarose", "Morrible", "Boq", "Dorothy")

parts = []
for targetfile in fullpaths:
    animal = targetfile.split('\\')[-3]
    session = targetfile.split('\\')[-2]
    try:
        file = read_lesion_monster_file(targetfile, animal, session)

        #df =  pd.DataFrame(file[0]['T_beam'][:,0], columns = ['time'])
        df =  pd.DataFrame(file[1]['t_beam'], columns = ['time'])
        df['events'] = file[0]['T_beam'][:,1]

        door = pd.DataFrame(file[1]['t_door'], columns=['time'])
        door['events'] = 'Door'
        entrance = pd.DataFrame(file[1]['t_enter'], columns=['time'])
        entrance['events'] = 'Entrance'
        reward = pd.DataFrame(file[1]['t_reward'], columns=['time'])
        reward['events'] = 'Reward'
        monster = pd.DataFrame(file[1]['t_monster'], columns=['time'])
        monster['events'] = 'Monster'
        exit = pd.DataFrame(file[1]['t_exit'], columns=['time'])
        exit['events'] = 'Exit'

        df = pd.concat([df, door, entrance, reward, monster, exit], ignore_index=True) 
        df = df.sort_values('time').reset_index(drop=True)

        m = df['events'].astype(str).str.strip().str.casefold().eq('door')
        df['trial'] = m.cumsum()
        df['time'] = (df['time'] - df['time'][0])/1000

        df['animal'] = animal
        df['session'] = session
        if animal in vsl:
            df['lesion'] = 'Lesion'
        else:
            df['lesion'] = 'Saline'
        
        parts.append(df)
        #successrate = len(df[df['events']=='Reward']) / len(df[df['events']=='Door'])

    except:
        print(targetfile)
mdf = pd.concat(parts)
mdf = mdf.sort_values(by=['animal', 'session', 'time'])
mdf['session_type'] = np.where(mdf['session'].str.contains('monster', case=False),'Monster','Reward')

mdf['session_n'] = mdf['session'].astype(str).str[-1].astype(int)

mdf['rewarded'] = (
    mdf.groupby(['animal', 'session', 'trial'])['events']
       .transform(lambda s: s.astype('string').str.contains('Reward', case=False, na=False).any())
       .map({True: 1, False: 0})
)

num = pd.to_numeric(mdf['events'], errors='coerce')

# map beambreaks to cm
conds = [
    (num < 4),
    (num < 8),
    (num < 16),
    (num < 32),
    (num < 64),
    (num < 128),
    (num < 196),
]
vals = [-1, 0, 5, 10, 20, 30, 40]
mapped = np.select(conds, vals, default=50)

# overwrite only numeric entries; strings stay as-is
mask = num.notna()
mdf.loc[mask, 'events'] = mapped[mask]

mdf['session_n'] = (mdf['session'].astype(str).str[-1].astype(int)-1)*10
mdf['trial_full'] = mdf['session_n'] + mdf['trial']

In [ ]:
successdf = mdf.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'rewarded'], as_index=False).head(1)[['animal', 'session_type', 'lesion', 'rewarded']]
successdf = successdf.groupby(['animal', 'session_type', 'lesion'], observed=True).agg("mean").reset_index()

successdf_lesion = successdf[(successdf['lesion']=='Lesion') & (successdf['session_type']=='Reward')].sort_values(['rewarded'], ascending = False)
successdf_saline = successdf[(successdf['lesion']=='Saline') & (successdf['session_type']=='Reward')].sort_values(['rewarded'], ascending = False)
successdf = mdf.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'session_n', 'rewarded'], as_index=False).head(1)[['animal', 'session_type', 'session_n','lesion', 'rewarded', 'trial', 'trial_full']]

successdf = mdf.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'session_n', 'rewarded'], as_index=False).head(1)[['animal', 'session_type', 'session_n','lesion', 'rewarded', 'trial', 'trial_full']]

Figure 2B

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import Normalize

lesion_order  = ["Saline", "Lesion"]
session_order = ["Reward", "Monster"]
vmin, vmax    = 0, 40
cmap          = "summer"
cbar_label    = "Max Distance (cm)"
figsize       = (10, 7.5)
savepath      = r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Behavior_LesionDistanceHeatmap.svg'

row_order_by_lesion = {
    "Saline": successdf_saline["animal"].drop_duplicates().tolist() if "successdf_saline" in globals() else [],
    "Lesion": successdf_lesion["animal"].drop_duplicates().tolist() if "successdf_lesion" in globals() else [],
}

df = trial_max.copy()
df["lesion"]       = pd.Categorical(df["lesion"], categories=lesion_order, ordered=True)
df["session_type"] = pd.Categorical(df["session_type"], categories=session_order, ordered=True)
df["trial_max_event"]      = pd.to_numeric(df["trial_max_event"], errors="coerce")

all_trials = np.sort(df["trial_full"].unique())

row_animals = {les: df.loc[df["lesion"] == les, "animal"].nunique() for les in lesion_order}
row_heights = [max(row_animals.get(les, 1), 1) for les in lesion_order]

fig = plt.figure(figsize=figsize)
gs  = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=row_heights,
    wspace=0.05, hspace=0.08
)

axes = {}
for r, les in enumerate(lesion_order):
    for c, ses in enumerate(session_order):
        ax = fig.add_subplot(gs[r, c])
        axes[(les, ses)] = ax

        sub = df[(df["lesion"] == les) & (df["session_type"] == ses)]

        mat = (
            sub.pivot_table(index="animal", columns="trial_full",
                            values="trial_max_event", aggfunc="mean")
               .reindex(columns=all_trials)
        )

        desired_animals = [a for a in row_order_by_lesion.get(les, []) if a in mat.index]
        if len(desired_animals):
            mat = mat.reindex(index=desired_animals)
        else:
            mat = mat.sort_index()

        sns.heatmap(
            mat,
            ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
            cbar=False,
            linewidths=0, linecolor=None,
            xticklabels=True, yticklabels=True
        )

        ax.set_xlabel("Trial" if r == 1 else "")
        ax.set_ylabel("Animal" if c == 0 else "")
        ax.tick_params(axis="x", labelrotation=0)

        if r == 0:
            ax.set_title(ses, pad=6, fontsize=12)

        if c == 1:
            ax.text(1.02, 0.5, les, transform=ax.transAxes, rotation=90,
                    va="center", ha="left", fontsize=11)

        for s in ax.spines.values():
            s.set_visible(False)

norm = Normalize(vmin=vmin, vmax=vmax)
sm   = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cax = fig.add_axes([0.92, 0.20, 0.02, 0.6])
fig.colorbar(sm, cax=cax, label=cbar_label)

plt.subplots_adjust(left=0.08, right=0.90, top=0.92, bottom=0.08)
plt.savefig(savepath)
plt.show()

Figure 2D

In [ ]:
%%R -i successdf -o emm
library(lme4)
library(emmeans)

successdf$animal<-as.factor(successdf$animal)
successdf$trial<-as.factor(successdf$trial)
successdf$session_n<-as.factor(successdf$session_n)

model <- glmer(rewarded ~ session_type*lesion + (1|animal) + (1|session_n) + (1|trial), data= successdf, family=binomial)

emm <- emmeans(model, ~ lesion * session_type, type = "response")
print(pairs(emm, by = 'session_type', adjust = "bonferroni"))  # lesion differences within session_type
print(pairs(emm, adjust = "bonferroni"))
emm <- as.data.frame(emm)

emm_maineffect <- emmeans(model, ~ session_type, type = "response")
print(pairs(emm_maineffect, adjust = "bonferroni"))  # lesion differences within session_type

z <- c(2.746, 10.840, 4.700, 5.897, 12.915, 1.394)
p.adjust(2 * pnorm(-abs(z)), method = "bonferroni")

In [ ]:

order         = ["Reward", "Monster"]
lesion_levels = ["Saline", "Lesion"]
dot_alpha     = 0.7
dot_size      = 40
jitter_width  = 0.08
dodge_width   = 0.22 
err_capsize   = 3
ylim          = (-0.05, 1.05)

successdf = mdf.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'rewarded'], as_index=False).head(1)[['animal', 'session_type', 'lesion', 'rewarded']]
successdf = successdf.groupby(['animal', 'session_type', 'lesion'], observed=True).agg("mean").reset_index()
successdf["session_type"] = pd.Categorical(successdf["session_type"], categories=order, ordered=True)
successdf = successdf[successdf["lesion"].isin(lesion_levels)]

emm_plot = emm.copy()

mean_col = next((c for c in ["emmean", "mean", "estimate", "prob"] if c in emm_plot.columns), None)
se_col   = next((c for c in ["SE", "se", "std.error"] if c in emm_plot.columns), None)
if mean_col is None or se_col is None:
    raise ValueError("`emm` must contain a mean column (emmean/mean/estimate) and a SE column (SE/se/std.error).")

emm_plot = emm_plot.rename(columns={mean_col: "mean", se_col: "se"})
emm_plot["session_type"] = pd.Categorical(emm_plot["session_type"], categories=order, ordered=True)
emm_plot = emm_plot[emm_plot["lesion"].isin(lesion_levels)]

emm_plot = emm_plot[["session_type", "lesion", "mean", "se"]].dropna()

fig, ax = plt.subplots(figsize=(6.8, 4.2))

x_positions = {sess: i for i, sess in enumerate(order)}
lesion_offset = {lesion_levels[0]: -dodge_width, lesion_levels[1]: dodge_width}

palette = {"Saline": "#4C78A8", "Lesion": "#F58518"}
for les in lesion_levels:
    sub = successdf[successdf["lesion"] == les]
    xs = sub["session_type"].map(x_positions).astype(float).to_numpy()
    xs = xs + lesion_offset[les] + (np.random.rand(len(xs)) - 0.5) * 2 * jitter_width
    ax.scatter(xs, sub["rewarded"], s=dot_size, alpha=dot_alpha, edgecolor="white", linewidth=0.6, c=palette[les], label=les)
for les in lesion_levels:
    subm = emm_plot[emm_plot["lesion"] == les].sort_values("session_type")
    if len(subm) == 0:
        continue
    xs = subm["session_type"].map(x_positions).astype(float).to_numpy() + lesion_offset[les]
    ax.errorbar(xs, subm["mean"], yerr=subm["se"],
                fmt="o", mfc="white", mec="black", ms=6,
                ecolor="black", elinewidth=1.4, capsize=err_capsize, linestyle="none", zorder=5)

ax.set_xticks(list(x_positions.values()), list(x_positions.keys()))
ax.set_xlim(-0.5, len(order) - 0.5)
ax.set_ylim(*ylim)
ax.set_ylabel("Fraction of Trials")
ax.set_xlabel("Session")
sns.despine()
handles, labels = ax.get_legend_handles_labels()
if handles:
    seen = set()
    uniq = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]

plt.tight_layout()
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Behavior_LesionSuccessRate.svg')
plt.show()

In [ ]:
successdf_p = successdf.pivot(index='animal', columns=['session_type'], values='rewarded').reset_index()
successdf_p['d_success'] = successdf_p['Monster'] - successdf_p['Reward']
successdf_p['lesion'] = np.where(successdf_p['animal'].isin(vsl), 'Lesion', 'Saline')

In [ ]:
%%R -i successdf_p

t.test(d_success ~ lesion, data = successdf_p)

In [ ]:
lesion_levels = ["Saline", "Lesion"]
dot_alpha     = 0.7
dot_size      = 40
jitter_width  = 0.10
err_capsize   = 3
palette       = {"Saline": "#4C78A8", "Lesion": "#F58518"}

df = successdf_p.copy()
need_cols = [c for c in ["animal", "lesion", "d_success"] if c in df.columns]
df = df[need_cols].rename(columns={"d_success": "d"})
df = df[df["lesion"].isin(lesion_levels)].dropna(subset=["d"])
df["lesion"] = pd.Categorical(df["lesion"], categories=lesion_levels, ordered=True)
summary = (
    df.groupby("lesion", observed=True)["d"]
      .agg(mean="mean", se="sem")
      .reindex(lesion_levels)
      .reset_index()
)

delta = np.nan
if summary["mean"].notna().all():
    try:
        delta = float(
            summary.set_index("lesion").loc["Lesion", "mean"]
            - summary.set_index("lesion").loc["Saline", "mean"]
        )
    except KeyError:
        pass

fig, ax = plt.subplots(figsize=(5.2, 4.2))
xmap = {"Saline": 0, "Lesion": 1}

rng = np.random.default_rng(42)
for les in lesion_levels:
    sub = df[df["lesion"] == les]
    if sub.empty:
        continue
    x0 = xmap[les]
    xs = x0 + (rng.random(len(sub)) - 0.5) * 2 * jitter_width
    ax.scatter(
        xs, sub["d"],
        s=dot_size, alpha=dot_alpha,
        edgecolor="white", linewidth=0.6,
        c=palette[les], label=les, zorder=2
    )

for _, row in summary.iterrows():
    les = row["lesion"]
    if pd.isna(row["mean"]):
        continue
    x0 = xmap[les]
    ax.errorbar(
        x0, row["mean"], yerr=row["se"],
        fmt="o", mfc="white", mec="black", ms=6,
        ecolor="black", elinewidth=1.4, capsize=err_capsize,
        linestyle="none", zorder=5
    )

ax.axhline(0, lw=1.0, color="k", alpha=0.25, zorder=1)
ax.set_xticks([xmap[k] for k in lesion_levels], lesion_levels)
ax.set_xlim(-0.5, 1.5)

if len(df):
    y_min, y_max = np.nanmin(df["d"]), np.nanmax(df["d"])
    if not np.isfinite(y_min): y_min = 0.0
    if not np.isfinite(y_max): y_max = y_min + 1.0
    pad = 0.07 * max(y_max - y_min, 1.0)
    ax.set_ylim(y_min - pad, y_max + pad)

ax.set_ylabel("Success Monster - Reward")
ax.set_xlabel("")
sns.despine()

plt.tight_layout()
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Behavior_LesionSuccessRateDiff.svg')
plt.show()

Figure 2F

In [ ]:
tmp = mdf.copy()
tmp["event_num"] = pd.to_numeric(tmp["events"], errors="coerce")

trial_max = (
    tmp.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'rewarded',"session_n", "trial_full"], observed=True)["event_num"]
       .max()
       .reset_index(name="trial_max_event")
)

trial_max['reached_end'] = np.where(trial_max['trial_max_event']>=40, 1, 0)

In [ ]:
%%R -i trial_max -o emm
library(survival)
library(coxme)


model<- coxph(Surv(trial_max_event, reached_end) ~ session_type * lesion + cluster(animal),
                      data = trial_max)
summary(model)

emm <- emmeans(model, ~ lesion | session_type)
print(pairs(emm, adjust = "bonferroni"))
emm <- as.data.frame(emm)


In [ ]:
order         = ["Reward", "Monster"]
lesion_levels = ["Saline", "Lesion"]
dot_alpha     = 0.7
dot_size      = 40
jitter_width  = 0.08
dodge_width   = 0.22
err_capsize   = 3
palette       = {"Saline": "#4C78A8", "Lesion": "#F58518"}

maxdf = (
    trial_max.dropna(subset=["trial_max_event"])
             .groupby(["animal","session_type","lesion"], observed=True)
             .agg(mean_max_event=("trial_max_event","mean"))
             .reset_index()
)

maxdf["session_type"] = pd.Categorical(maxdf["session_type"], categories=order, ordered=True)
maxdf = maxdf[maxdf["lesion"].isin(lesion_levels)]

summary = (
    maxdf.groupby(["session_type","lesion"], observed=True)["mean_max_event"]
         .agg(mean="mean", se="sem")
         .reset_index()
)
summary["session_type"] = pd.Categorical(summary["session_type"], categories=order, ordered=True)
summary = summary[summary["lesion"].isin(lesion_levels)]

fig, ax = plt.subplots(figsize=(6.8, 4.2))

x_positions   = {sess: i for i, sess in enumerate(order)}
lesion_offset = {lesion_levels[0]: -dodge_width, lesion_levels[1]: dodge_width}

rng = np.random.default_rng(42)
labeled = set()
for les in lesion_levels:
    sub = maxdf[maxdf["lesion"] == les]
    if sub.empty:
        continue
    xs = sub["session_type"].map(x_positions).astype(float).to_numpy()
    xs = xs + lesion_offset[les] + (rng.random(len(xs)) - 0.5) * 2 * jitter_width
    lab = les if les not in labeled else None
    ax.scatter(
        xs, sub["mean_max_event"],
        s=dot_size, alpha=dot_alpha,
        edgecolor="white", linewidth=0.6,
        c=palette[les], label=lab, zorder=2
    )
    labeled.add(les)

for les in lesion_levels:
    subm = summary[summary["lesion"] == les].sort_values("session_type")
    if len(subm) == 0:
        continue
    xs = subm["session_type"].map(x_positions).astype(float).to_numpy() + lesion_offset[les]
    ax.errorbar(
        xs, subm["mean"], yerr=subm["se"],
        fmt="o", mfc="white", mec="black", ms=6,
        ecolor="black", elinewidth=1.4, capsize=err_capsize,
        linestyle="none", zorder=5
    )

ax.set_xticks(list(x_positions.values()), list(x_positions.keys()))
ax.set_xlim(-0.5, len(order) - 0.5)

if len(maxdf):
    y_min = np.nanmin(maxdf["mean_max_event"])
    y_max = np.nanmax(maxdf["mean_max_event"])
    if not np.isfinite(y_min): y_min = 0.0
    if not np.isfinite(y_max): y_max = y_min + 1.0
    pad = 0.05 * max(y_max - y_min, 1.0)
    ax.set_ylim(y_min - pad, y_max + pad)

ax.set_ylabel("Distance (cm)")
ax.set_xlabel("Session")
sns.despine()

plt.tight_layout()
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Behavior_LesionReturnPoint.svg')
plt.show()

In [ ]:
maxdf_p = maxdf.pivot(index='animal', columns=['session_type'], values='mean_max_event').reset_index()
maxdf_p['d_distance'] = maxdf_p['Monster'] - maxdf_p['Reward']
maxdf_p['lesion'] = np.where(maxdf_p['animal'].isin(vsl), 'Lesion', 'Saline')

In [ ]:
%%R -i maxdf_p

t.test(d_distance ~ lesion, data = maxdf_p)

In [ ]:
lesion_levels = ["Saline", "Lesion"]
dot_alpha     = 0.7
dot_size      = 40
jitter_width  = 0.10
err_capsize   = 3
palette       = {"Saline": "#4C78A8", "Lesion": "#F58518"}

df = maxdf_p.copy()

need_cols = [c for c in ["animal", "lesion", "d_distance"] if c in df.columns]
df = df[need_cols].rename(columns={"d_distance": "d"})
df = df[df["lesion"].isin(lesion_levels)].dropna(subset=["d"])
df["lesion"] = pd.Categorical(df["lesion"], categories=lesion_levels, ordered=True)

summary = (
    df.groupby("lesion", observed=True)["d"]
      .agg(mean="mean", se="sem")
      .reindex(lesion_levels)
      .reset_index()
)

delta = np.nan
if summary["mean"].notna().all():
    try:
        delta = float(
            summary.set_index("lesion").loc["Lesion", "mean"]
            - summary.set_index("lesion").loc["Saline", "mean"]
        )
    except KeyError:
        pass

fig, ax = plt.subplots(figsize=(5.2, 4.2))
xmap = {"Saline": 0, "Lesion": 1}

rng = np.random.default_rng(42)
for les in lesion_levels:
    sub = df[df["lesion"] == les]
    if sub.empty:
        continue
    x0 = xmap[les]
    xs = x0 + (rng.random(len(sub)) - 0.5) * 2 * jitter_width
    ax.scatter(
        xs, sub["d"],
        s=dot_size, alpha=dot_alpha,
        edgecolor="white", linewidth=0.6,
        c=palette[les], label=les, zorder=2
    )

for _, row in summary.iterrows():
    les = row["lesion"]
    if pd.isna(row["mean"]):
        continue
    x0 = xmap[les]
    ax.errorbar(
        x0, row["mean"], yerr=row["se"],
        fmt="o", mfc="white", mec="black", ms=6,
        ecolor="black", elinewidth=1.4, capsize=err_capsize,
        linestyle="none", zorder=5
    )

ax.axhline(0, lw=1.0, color="k", alpha=0.25, zorder=1)
ax.set_xticks([xmap[k] for k in lesion_levels], lesion_levels)
ax.set_xlim(-0.5, 1.5)

if len(df):
    y_min, y_max = np.nanmin(df["d"]), np.nanmax(df["d"])
    if not np.isfinite(y_min): y_min = 0.0
    if not np.isfinite(y_max): y_max = y_min + 1.0
    pad = 0.07 * max(y_max - y_min, 1.0)
    ax.set_ylim(y_min - pad, y_max + pad)

ax.set_ylabel("Distance Monster - Reward")
ax.set_xlabel("")
sns.despine()

plt.tight_layout()
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Behavior_LesionReturnPointDiff.svg')
plt.show()

Figure 2G

In [ ]:
#Calculate retreat rate (quit before 30cm) per session for each animal
trial_max_retreat = trial_max.copy()
trial_max_retreat['retreat_before_30cm'] = np.where(trial_max_retreat['trial_max_event'] < 30, 1, 0)

retreat_by_session = (
trial_max_retreat.groupby(['animal', 'session', 'session_type', 'lesion'], observed=True)
.agg(
n_trials=('retreat_before_30cm', 'count'),
n_retreats=('retreat_before_30cm', 'sum'),
retreat_rate=('retreat_before_30cm', 'mean')
)
.reset_index()
)

df_retreat = retreat_by_session.copy()
sessions_to_plot = ["Control_1", "Monster_1", "Monster_2", "Monster_3"]
df_retreat = df_retreat[df_retreat["session"].isin(sessions_to_plot)]

retreat_monster = df_retreat[df_retreat['session_type'] == 'Monster']
retreat_monster['session_num'] = retreat_monster['session'].str.extract(r'_(\d+)').astype(int)

In [ ]:
%%R -i retreat_monster 

library(lmerTest)
library(emmeans)

retreat_monster$animal <- as.factor(retreat_monster$animal)
retreat_monster$session_num <- as.numeric(retreat_monster$session_num)  # numeric for linear trend
retreat_monster$lesion <- as.factor(retreat_monster$lesion)

model <- lmer(
  retreat_rate ~ session_num * lesion + (1 | animal),
  data = retreat_monster
)

summary(model)

# Extract fixed effects coefficients
coefs <- summary(model)$coefficients
cat("\n===== FIXED EFFECTS (Beta Weights) =====\n")
print(coefs)

confint(model, method = "Wald")

In [ ]:
# Set publication-quality style
sns.set_style("ticks")
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42  
# Make sure session_num exists
if 'session_num' not in retreat_monster.columns:
    retreat_monster['session_num'] = retreat_monster['session'].str.extract(r'_(\d+)').astype(int)
# Calculate mean and SEM for each group at each session
summary = retreat_monster.groupby(['lesion', 'session_num'])['retreat_rate'].agg(['mean', 'sem']).reset_index()
# Create the plot
fig, ax = plt.subplots(figsize=(4, 3))  
colors = {'Saline': '#000000', 'Lesion': '#808080'}  
markers = {'Saline': 'o', 'Lesion': 's'}  # Circle and square
for lesion_type in ['Saline', 'Lesion']:
    data = summary[summary['lesion'] == lesion_type]
    
    # Plot line with error bars
    ax.errorbar(data['session_num'], data['mean'], yerr=data['sem'],
                label=lesion_type, color=colors[lesion_type], 
                marker=markers[lesion_type], markersize=6,
                linewidth=1.5, capsize=4, capthick=1.5,
                markerfacecolor='white' if lesion_type == 'Lesion' else colors[lesion_type],
                markeredgewidth=1.5)
# Formatting
ax.set_xlabel('Monster Session Number', fontsize=11)
ax.set_ylabel('Predictive Avoidance Rate', fontsize=11)
ax.set_xticks(range(1, 4))
ax.set_xticklabels(['1', '2', '3'])
ax.legend(title='', fontsize=9, frameon=False, loc='best')
ax.set_ylim([0, max(summary['mean']) * 1.15])
# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Make remaining spines thinner
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
# Adjust tick parameters
ax.tick_params(width=1, length=4)
plt.tight_layout()
savepath = Path.home() / "Uchida Lab Dropbox" / "Isobel Green" / "Isobel_Eshaan" / "Code" / "Figure Mockup" / "Isobel_SVGs" / "PredictiveAvoidance_Monster.svg"
savepath.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(savepath, format='svg', bbox_inches='tight')

plt.show()

# Print summary statistics
print("\nPredictive Avoidance Rate Summary:")
print(summary.round(3))